# TS Convergence: Single-Structure Demo

Demonstrates `phase_diagram_workflows.free_energies.ts_convergence.single` --
the temperature-scaling (TS) bracket-convergence logic for **one structure at
a time** (multi-structure sweeps are a separate, not-yet-built layer on top
of this).

Two ways to drive a bracket to convergence, both shown below:

1. **Manual**: `refine_temperature_bracket_manually` -- call it again
   yourself each time you want to check in; it submits/narrows on your
   behalf but never blocks and never chains on its own.
2. **Dependency chain**: `submit_bracket_chain` -- submit once and walk
   away; the full candidate bracket sequence is submitted up front to
   *whichever executor you hand it*, each step wired to depend on the
   previous one's Future, and executorlib itself resolves that chain
   (natively supported by `SingleNodeExecutor`, `SlurmClusterExecutor`, and
   `FluxClusterExecutor` alike -- we use `SingleNodeExecutor` below only
   because that's what runs locally).

Both read/write a `bracket_log.csv` dataframe per structure (via
`load_bracket_history`) as the record of what's already been tried -- this
is what lets a later call (even in a new session, at a different tolerance)
resume correctly without depending on the executor's own result caching.


In [1]:
import sys
from pathlib import Path

# Add project root to path for imports
PROJECT_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "pyproject.toml").is_file()),
    Path.cwd(),
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from phase_diagram_workflows.free_energies.ts_convergence.single import (
    refine_temperature_bracket_manually,
    submit_bracket_chain,
    load_bracket_history,
)
from executorlib import SingleNodeExecutor

EXAMPLE_ROOT = PROJECT_ROOT / "notebooks" / "ts_convergence_single_demo"


In [2]:
import os
import time
import pandas as pd
from ase.build import bulk
from lammpsparser import get_potential_by_name


## 1. Structure and Potential Setup

Small cell (32 atoms) and modest equilibration/switching steps, purely so this demo runs quickly.

In [3]:
# Small Al structure -- fast, so this demo runs in a couple of minutes
structure = bulk("Al", cubic=True).repeat(2)
print(f"Created Al structure with {len(structure)} atoms")


Created Al structure with 32 atoms


In [4]:
potential_df = get_potential_by_name("1999--Mishin-Y--Al--LAMMPS--ipr1")
potential_df = potential_df.to_frame().transpose()
print("Potential loaded successfully:")
display(potential_df.head())


Potential loaded successfully:


/cmmc/ptmp/pyironhb/pyiron_latest_env/lib/python3.12/site-packages/lammpsparser/potential.py:397: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pot["Config"] = config_lst


,Config,Filename,Model,Name,Species,Citations
49,"[pair_style eam/alloy, pair_coeff * * /cmmc/pt...",[potential_LAMMPS/1999--Mishin-Y--Al--LAMMPS--...,NISTiprpy,1999--Mishin-Y--Al--LAMMPS--ipr1,[Al],[{'Mishin_1999': {'title': 'Interatomic potent...


In [5]:
# ts mode: forward/backward reversible scaling over one temperature bracket.
# Step counts kept small (demo speed only -- not production-quality TI).
ts_params = {
    "mode": "ts",
    "pressure": 0,
    "n_equilibration_steps": 100,
    "n_switching_steps": 100,
    "n_print_steps": 25,
    "equilibration_control": "berendsen",
    "md": {"thermostat_damping": 0.5},
    "tolerance": {"spring_constant": 0.01, "pressure": 0.5},
    "queue": {"cores": 1, "scheduler": "local"},
    "reference_phase": "solid",
    "file_format": "lammps-data",
}


## 2. Manual mode: `refine_temperature_bracket_manually`

Each call checks disk (via the bracket log), and either reports a result or
submits/narrows and returns immediately. Here we just call it in a polling
loop and print what comes back -- that loop is exactly what a caller has to
provide themselves in this mode.

Using a deliberately unreachable tolerance (`-1.0` -- the criterion is an
absolute value, so it's always >= 0 and this can never be satisfied) so we
actually see a `"resubmitted"` narrowing happen, not just
`"pending" -> "converged"`.

In [6]:
print("Setting up executor...")
manual_working_directory_root = str(EXAMPLE_ROOT / "manual")
executor = SingleNodeExecutor(hostname_localhost=True, max_cores=1)
print("Executor ready!")


Setting up executor...


Executor ready!


In [7]:
# A real calphy run here (100/100 steps on 32 atoms) takes on the order of
# 10-20s wall time including executor/LAMMPS startup -- poll patiently, and
# don't stop on "incomplete": that status can also mean "read the directory
# mid-write", which resolves itself on the next poll once calphy finishes.
result = {"status": "pending"}
for elapsed in range(0, 200, 10):
    if result["status"] not in ("pending", "incomplete"):
        break
    result = refine_temperature_bracket_manually(
        input_structure=structure,
        calphy_parameters=ts_params,
        potential_df=potential_df,
        executor=executor,
        working_directory_root=manual_working_directory_root,
        initial_bracket=(700.0, 720.0),
        tolerance=-1.0,  # deterministically unreachable -- forces a narrowing step
        step_upper=10.0,
    )
    print(f"[{elapsed}s] {result['status']} {result['bracket']}")
    if result["status"] in ("pending", "incomplete"):
        time.sleep(10)

print("\nFinal status:", result["status"], "at bracket", result["bracket"])
executor.shutdown(wait=False, cancel_futures=False)


[0s] pending (700.0, 720.0)


/cmmc/ptmp/pyironhb/pyiron_latest_env/lib/python3.12/site-packages/ase/io/lammpsdata.py:72: FutureWarning: "style" is deprecated; please use "atom_style".
  warnings.warn(


[10s] incomplete (700.0, 720.0)


[20s] incomplete (700.0, 720.0)


Input validation successful. Proceeding with calculation.


[30s] resubmitted (700.0, 710.0)

Final status: resubmitted at bracket (700.0, 710.0)


In [8]:
# The bracket log this call read/wrote -- a plain dataframe, not an
# executor-specific cache, so it's just as readable outside this session.
bracket_log = pd.read_csv(os.path.join(manual_working_directory_root, "bracket_log.csv"))
bracket_log


,t_low,t_high,criterion
0,700.0,720.0,0.005198
1,700.0,710.0,NaN


## 3. Dependency chain: `submit_bracket_chain`

The full candidate bracket sequence (`(700, 720) -> (700, 715) -> (700, 710)`
for the settings below -- `step_upper=5.0`, `max_iterations=2`) is decided
up front and submitted all at once: the first bracket with no dependency,
each later one depending on the Future returned for the bracket before it.
Nothing in this notebook calls anything a second time -- the chain is driven
entirely by executorlib's own dependency resolution.

`submit_bracket_chain` never constructs an executor itself -- it only ever
calls `.submit()` on the exact object you pass it. To make that concrete
(not just asserted), the executor below is wrapped in a tiny counter that
records every `.submit()` call made on it, so we can confirm afterward that
every step in the chain really did go through *this one object* -- swap it
for a `SlurmClusterExecutor` or `FluxClusterExecutor` and nothing here would
need to change.

In [9]:
class SubmitCountingExecutor:
    """Thin pass-through wrapper: proves submit_bracket_chain only ever
    calls .submit() on the exact executor object handed to it, whatever
    concrete executor that is (SingleNodeExecutor here; a SlurmClusterExecutor
    or FluxClusterExecutor would work identically since submit_bracket_chain
    only relies on the generic submit(fn, **kwargs) -> Future contract plus
    the dependency-resolution behavior all three provide by default).
    """

    def __init__(self, wrapped):
        self._wrapped = wrapped
        self.submit_count = 0

    def submit(self, fn, **kwargs):
        self.submit_count += 1
        return self._wrapped.submit(fn, **kwargs)

    def shutdown(self, *args, **kwargs):
        return self._wrapped.shutdown(*args, **kwargs)


auto_working_directory_root = str(EXAMPLE_ROOT / "bracket_chain")
real_executor = SingleNodeExecutor(hostname_localhost=True, max_cores=1)
my_executor = SubmitCountingExecutor(real_executor)  # <-- bring your own executor


In [10]:
futures, working_directory = submit_bracket_chain(
    input_structure=structure,
    calphy_parameters=ts_params,
    potential_df=potential_df,
    executor=my_executor,  # the exact object constructed above -- nothing else submits anything
    working_directory_root=auto_working_directory_root,
    initial_bracket=(700.0, 720.0),
    tolerance=-1.0,  # deterministically unreachable -- forces every hop to actually run
    step_upper=5.0,
    max_iterations=2,
)

print(f"{len(futures)} brackets precomputed and submitted up front")

# Waiting on the LAST future is only meaningful if executorlib's dependency
# resolution genuinely drove every earlier step to completion first --
# that's exactly what's being demonstrated here.
last_result = futures[-1].result()
print("Last bracket in the chain:", last_result["bracket"], "status:", last_result["status"])

print(f"\nour executor object handled {my_executor.submit_count} of {len(futures)} submissions")
real_executor.shutdown(wait=False, cancel_futures=False)


3 brackets precomputed and submitted up front


/cmmc/ptmp/pyironhb/pyiron_latest_env/lib/python3.12/site-packages/ase/io/lammpsdata.py:72: FutureWarning: "style" is deprecated; please use "atom_style".
  warnings.warn(


/cmmc/ptmp/pyironhb/pyiron_latest_env/lib/python3.12/site-packages/ase/io/lammpsdata.py:72: FutureWarning: "style" is deprecated; please use "atom_style".
  warnings.warn(


Input validation successful. Proceeding with calculation.
Input validation successful. Proceeding with calculation.


/cmmc/ptmp/pyironhb/pyiron_latest_env/lib/python3.12/site-packages/ase/io/lammpsdata.py:72: FutureWarning: "style" is deprecated; please use "atom_style".
  warnings.warn(


Input validation successful. Proceeding with calculation.


/cmmc/ptmp/pyironhb/pyiron_latest_env/lib/python3.12/site-packages/ase/io/lammpsdata.py:72: FutureWarning: "style" is deprecated; please use "atom_style".
  warnings.warn(


Last bracket in the chain: (700.0, 710.0) status: narrowed

our executor object handled 3 of 3 submissions
Input validation successful. Proceeding with calculation.


In [11]:
auto_bracket_log = pd.read_csv(os.path.join(auto_working_directory_root, "bracket_log.csv"))
auto_bracket_log


,t_low,t_high,criterion
0,700.0,720.0,0.011369
1,700.0,715.0,0.002057
2,700.0,710.0,0.008159


## 4. Resuming from the log

`load_bracket_history` is what both modes call internally to figure out
what's already been tried -- reading `bracket_log.csv` first, falling back
to rescanning calphy output directories only if that log is missing. Calling
it directly shows what a fresh session (e.g. asking to rerun at a different
temperature range) would see without doing any new calphy work.

In [12]:
for label, root in [("manual", manual_working_directory_root), ("bracket_chain", auto_working_directory_root)]:
    tried_brackets, criteria_by_bracket = load_bracket_history(root)
    print(f"{label}: tried={sorted(tried_brackets)}")
    print(f"{label}: criteria={criteria_by_bracket}")


manual: tried=[(700.0, 710.0), (700.0, 720.0)]
manual: criteria={(700.0, 720.0): 0.00519767132403}
bracket_chain: tried=[(700.0, 710.0), (700.0, 715.0), (700.0, 720.0)]
bracket_chain: criteria={(700.0, 720.0): 0.01136932068469, (700.0, 715.0): 0.0020572497237099, (700.0, 710.0): 0.0081586593103102}
